# Final export: test example

Shows what `final_export.py` produces for a small selection: genomes from the three databases (NCBI, Ensembl, GWH), plus two that fail.
For each species, the file contains its genomes with the links to their files, and its representative raw data runs. Genomes and runs are joined by the taxid of the species.
The result is one JSON file for the whole selection, written next to this notebook (`example_final_export.json`). Nothing is downloaded, only the links.

In [1]:
import sys
sys.path.append("../../..")

import json
from backend.final_export.final_export import build_final_export, write_export

## 1. The selection

Each genome says which database it comes from, its accession and its taxid (`organism.tax_id` at NCBI, `taxonomy_id` at Ensembl and GWH).
The cat has two genomes (NCBI RefSeq and Ensembl), to show that its raw data is searched only once.
The last two are there on purpose: an accession that doesn't exist, and a genome without taxid.

In [2]:
genomes = [
    {"source": "NCBI", "accession": "GCF_018350175.1", "taxid": 9685},
    {"source": "Ensembl", "accession": "GCA_018350175.1", "taxid": 9685, "species": "Felis catus"},
    {"source": "NCBI", "accession": "GCF_014176215.1", "taxid": 9407, "assembly_name": "mRouAeg1.p", "species": "Rousettus aegyptiacus"},
    {"source": "GWH", "accession": "GWHBHRQ00000000", "taxid": 61621},
    {"source": "NCBI", "accession": "GCF_999999999.1", "taxid": 9685},
    {"source": "NCBI", "accession": "GCF_000002285.5"},
]

## 2. Build the export

The genomes are grouped by species, then the links of each genome are collected, then the raw data of each species (RNA-seq by default).
It usually takes less than a minute, but it can be much slower when ENA's server is slow.

In [3]:
export = build_final_export(genomes)

print(len(export["species"]), "species,", len(export["problems"]), "problems")
for species in export["species"]:
    print()
    print(species["taxid"], species["species"])
    for genome in species["genomes"]:
        found = sum(link is not None for link in genome["links"].values())
        print(f'  genome   {genome["source"]:8} {genome["accession"]:16} {found} files')
    for raw in species["raw_data"]:
        print(f'  raw data {raw["data_type"]:8} {raw["total_runs"]} runs in ENA, {len(raw["runs"])} chosen')

3 species, 2 problems

9685 Felis catus
  genome   NCBI     GCF_018350175.1  11 files
  genome   Ensembl  GCA_018350175.1  6 files
  raw data RNA-Seq  2010 runs in ENA, 10 chosen

9407 Rousettus aegyptiacus
  genome   NCBI     GCF_014176215.1  11 files
  raw data RNA-Seq  361 runs in ENA, 10 chosen

61621 Rhinopithecus bieti
  genome   GWH      GWHBHRQ00000000  7 files
  raw data RNA-Seq  12 runs in ENA, 10 chosen


## 3. The problems

Each problem says at which step it happened: "input" (something missing in the selection), "links" or "raw data".
The other genomes and species are not affected: the cat still has its other two genomes and its runs.

In [4]:
for problem in export["problems"]:
    print(problem["step"], "|", problem.get("accession") or problem.get("taxid"), "->", problem["message"])

input | GCF_000002285.5 -> no taxid given, so the genome can't be joined to its species
links | GCF_999999999.1 -> GCF_999999999.1 was not found on NCBI


## 4. How the runs were chosen

The file says which method and which settings were used for the raw data, so anyone reading it knows how the runs were chosen.

In [5]:
print(json.dumps(export["raw_data_selection"], indent=2))

{
  "method": "rules",
  "source": "ENA (also contains the runs of SRA and DDBJ)",
  "data_types": [
    "RNA-Seq"
  ],
  "max_runs": 10,
  "min_bases": 1000000000,
  "excluded_sources": [
    "GENOMIC SINGLE CELL",
    "METAGENOMIC",
    "METATRANSCRIPTOMIC",
    "TRANSCRIPTOMIC SINGLE CELL"
  ]
}


## 5. Write the JSON file


In [6]:
write_export(export, "example_final_export.json")
print(json.dumps(export, indent=2)[:3000])  # the beginning of the file

{
  "format": "genomexplore-export",
  "format_version": "1.0",
  "created": "2026-11-12T14:23:02",
  "raw_data_selection": {
    "method": "rules",
    "source": "ENA (also contains the runs of SRA and DDBJ)",
    "data_types": [
      "RNA-Seq"
    ],
    "max_runs": 10,
    "min_bases": 1000000000,
    "excluded_sources": [
      "GENOMIC SINGLE CELL",
      "METAGENOMIC",
      "METATRANSCRIPTOMIC",
      "TRANSCRIPTOMIC SINGLE CELL"
    ]
  },
  "species": [
    {
      "taxid": "9685",
      "species": "Felis catus",
      "genomes": [
        {
          "source": "NCBI",
          "accession": "GCF_018350175.1",
          "species": "Felis catus",
          "links": {
            "genome_fasta": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/018/350/175/GCF_018350175.1_F.catus_Fca126_mat1.0/GCF_018350175.1_F.catus_Fca126_mat1.0_genomic.fna.gz",
            "gff": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/018/350/175/GCF_018350175.1_F.catus_Fca126_mat1.0/GCF_018350175.1_F.cat